# SimpleObjectCloudEncoder

A stripped-down PTv1-style encoder for semantic object clouds in object goal navigation.

Each input object is a **class index** in `[0, num_classes)` (default 21) and a **3D position**.
The encoder produces a fixed-size **scene summary (CLS) embedding** of dimension `out_dim` (default 32).

## Architecture

1. `ClassEmbedding`: `nn.Embedding(num_classes, d_model)` followed by `LayerNorm`.
2. Learnable CLS token, positioned at `agent_pos` for egocentric RPE.
3. `L=2` `SimpleObjectCloudLayer`s, each doing:
    1. **Object self-attention** (PTv1 full vector attention with additive RPE).
    2. **CLS cross-attention** (CLS queries the objects; objects are not updated).
    3. Pre-norm FFN on objects.
    4. Pre-norm FFN on CLS.
4. Final `LayerNorm` + `Linear(d_model, out_dim)` head on CLS.

## PTv1 vector attention

Per query `i` and key `j`, with `\delta` derived from the relative position `p_i - p_j`:

$$
\delta_{ij} = \mathrm{MLP_{pos}}(p_i - p_j) \qquad
r_{ij} = \gamma\!\bigl(Q_i - K_j + \delta_{ij}\bigr) \qquad
\alpha_{ij} = \mathrm{softmax}_j(r_{ij}) \qquad
y_i = \sum_j \alpha_{ij} \odot (V_j + \delta_{ij})
$$

`\gamma` is a 2-layer GELU MLP `d \to d \to d`. Softmax is taken **per feature channel** over the keys, giving the characteristic "vector" attention.

## Differences vs the original `point_transformer_exploration.ipynb`

Removed (PTv2/v3 efficiency tricks not needed at our scale):
- Grouped vector attention and `num_groups`.
- The PTv2 RPE multiplier branch (`r_mult`).
- The two-stream CLIP/DINOv2 input projection and L2 normalization.

Kept (still useful):
- Pre-norm structure with separate norms for cross-attention.
- Padding-mask handling.
- Read-only CLS cross-attention.
- All four `compute_relative_positions` modes.

## Coordinate convention

The model is **agnostic** to the input frame, but for object goal navigation the recommended convention is **agent-frame** coordinates:

$$
\mathrm{obj\_pos} = R_\mathrm{agent}^{\top}\,(\mathrm{obj\_pos}_\mathrm{world} - \mathrm{agent\_pos}_\mathrm{world}), \qquad \mathrm{agent\_pos} = \mathbf{0}
$$

This makes RPE encode "object is forward / left / right of me" rather than world-axis directions, which solves the heading-blindness issue without any model change.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Literal

## Relative position encoding helper

In [ ]:
RPEMode = Literal["raw", "normalized", "direction_only", "log_distance"]


def compute_relative_positions(
    pos_i: torch.Tensor,                          # (B, N, 3)
    pos_j: torch.Tensor,                          # (B, M, 3)
    mode: RPEMode = "log_distance",
    scene_scale: Optional[torch.Tensor] = None,   # (B,) only used for "normalized"
) -> torch.Tensor:
    """Relative position vectors from every point in ``pos_i`` to every point in ``pos_j``.

    Returns a tensor of shape ``(B, N, M, 3)``.

    Modes
    -----
    raw            : metric offset ``p_i - p_j``.
    direction_only : unit vector, no distance information.
    log_distance   : unit vector scaled by ``log(1 + ||offset||)`` (default).
    normalized     : offset divided by per-batch scene scale.
    """
    delta = pos_i.unsqueeze(2) - pos_j.unsqueeze(1)         # (B, N, M, 3)

    if mode == "raw":
        return delta

    dist = delta.norm(dim=-1, keepdim=True).clamp(min=1e-6)  # (B, N, M, 1)
    direction = delta / dist                                  # (B, N, M, 3)

    if mode == "direction_only":
        return direction

    if mode == "log_distance":
        return direction * torch.log1p(dist)

    if mode == "normalized":
        if scene_scale is None:
            scene_scale = dist.flatten(1).max(dim=1).values.clamp(min=1e-6)
        return delta / scene_scale.view(-1, 1, 1, 1)

    raise ValueError(f"Unknown RPE mode: {mode}")

## PTv1 vector attention

Full vector attention with additive RPE. Used for both the object self-attention and the CLS cross-attention; cross-attention sets `use_separate_norms=True` so the query (CLS) and key/value (objects) streams are normalized independently.

In [ ]:
class VectorAttention(nn.Module):
    """PTv1-style full vector attention with additive RPE and pre-norm.

    For each query ``i`` and key ``j``::

        delta_ij  = MLP_pos(p_i - p_j)                       # (B, N, M, d)
        r_ij      = gamma(Q_i - K_j + delta_ij)              # (B, N, M, d)
        alpha_ij  = softmax_j(r_ij)                          # vector softmax over keys
        y_i       = sum_j(alpha_ij * (V_j + delta_ij))       # (B, N, d)
    """

    def __init__(
        self,
        d_model: int,
        rpe_mode: RPEMode = "log_distance",
        dropout: float = 0.0,
        use_separate_norms: bool = False,
    ):
        super().__init__()
        self.d_model = d_model
        self.rpe_mode = rpe_mode

        self.norm_q = nn.LayerNorm(d_model)
        self.norm_kv = nn.LayerNorm(d_model) if use_separate_norms else self.norm_q

        self.proj_q = nn.Linear(d_model, d_model, bias=False)
        self.proj_k = nn.Linear(d_model, d_model, bias=False)
        self.proj_v = nn.Linear(d_model, d_model, bias=False)
        self.proj_out = nn.Linear(d_model, d_model)

        self.pos_mlp = nn.Sequential(
            nn.Linear(3, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

        self.gamma = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

        self.attn_dropout = nn.Dropout(dropout)
        self.out_dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,                              # (B, N, d)
        pos: torch.Tensor,                            # (B, N, 3)
        key_x: torch.Tensor,                          # (B, M, d)
        key_pos: torch.Tensor,                        # (B, M, 3)
        mask: Optional[torch.Tensor] = None,          # (B, M) bool, True = valid
        scene_scale: Optional[torch.Tensor] = None,   # (B,) for "normalized" RPE
    ) -> torch.Tensor:                                # (B, N, d)
        x_norm = self.norm_q(x)
        kv_norm = self.norm_kv(key_x)

        Q = self.proj_q(x_norm)                       # (B, N, d)
        K = self.proj_k(kv_norm)                      # (B, M, d)
        V = self.proj_v(kv_norm)                      # (B, M, d)

        rel = compute_relative_positions(pos, key_pos, self.rpe_mode, scene_scale)
        delta = self.pos_mlp(rel)                     # (B, N, M, d)

        relation = self.gamma(Q.unsqueeze(2) - K.unsqueeze(1) + delta)  # (B, N, M, d)

        if mask is not None:
            # mask: (B, M) -> (B, 1, M, 1), broadcast over queries and channels
            relation = relation.masked_fill(
                ~mask.unsqueeze(1).unsqueeze(-1), float("-inf")
            )

        attn = F.softmax(relation, dim=2)             # (B, N, M, d) per-channel softmax over keys
        attn = self.attn_dropout(attn)

        Vp = V.unsqueeze(1) + delta                   # (B, N, M, d)
        out = (attn * Vp).sum(dim=2)                  # (B, N, d)
        out = self.out_dropout(self.proj_out(out))

        return x + out

## Feed-forward block

In [ ]:
class FFN(nn.Module):
    """Pre-norm feed-forward with residual."""

    def __init__(self, d_model: int, expansion: int = 2, dropout: float = 0.0):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.net = nn.Sequential(
            nn.Linear(d_model, d_model * expansion),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * expansion, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.net(self.norm(x))

## Class embedding

In [ ]:
class ClassEmbedding(nn.Module):
    """Categorical class index -> ``d_model`` vector with LayerNorm.

    Inputs:  ``class_idx`` int64 tensor of shape ``(B, N)`` with values in
             ``[0, num_classes)``.
    Outputs: float tensor of shape ``(B, N, d_model)``.
    """

    def __init__(self, num_classes: int, d_model: int):
        super().__init__()
        self.embed = nn.Embedding(num_classes, d_model)
        self.norm = nn.LayerNorm(d_model)
        nn.init.normal_(self.embed.weight, std=0.02)

    def forward(self, class_idx: torch.Tensor) -> torch.Tensor:
        return self.norm(self.embed(class_idx))

## Encoder layer

In [ ]:
class SimpleObjectCloudLayer(nn.Module):
    """One encoder layer:

    1. Object self-attention (objects -> objects).
    2. CLS cross-attention (CLS queries objects; objects unchanged).
    3. FFN on objects.
    4. FFN on CLS.
    """

    def __init__(
        self,
        d_model: int,
        rpe_mode: RPEMode = "log_distance",
        ffn_expansion: int = 2,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.obj_self_attn = VectorAttention(
            d_model, rpe_mode, dropout, use_separate_norms=False,
        )
        self.cls_cross_attn = VectorAttention(
            d_model, rpe_mode, dropout, use_separate_norms=True,
        )
        self.obj_ffn = FFN(d_model, ffn_expansion, dropout)
        self.cls_ffn = FFN(d_model, ffn_expansion, dropout)

    def forward(
        self,
        obj_feat: torch.Tensor,                       # (B, N, d)
        obj_pos: torch.Tensor,                        # (B, N, 3)
        cls_feat: torch.Tensor,                       # (B, 1, d)
        cls_pos: torch.Tensor,                        # (B, 1, 3)
        obj_mask: Optional[torch.Tensor] = None,      # (B, N) bool
        scene_scale: Optional[torch.Tensor] = None,
    ):
        obj_feat = self.obj_self_attn(
            x=obj_feat, pos=obj_pos,
            key_x=obj_feat, key_pos=obj_pos,
            mask=obj_mask, scene_scale=scene_scale,
        )
        cls_feat = self.cls_cross_attn(
            x=cls_feat, pos=cls_pos,
            key_x=obj_feat, key_pos=obj_pos,
            mask=obj_mask, scene_scale=scene_scale,
        )
        obj_feat = self.obj_ffn(obj_feat)
        cls_feat = self.cls_ffn(cls_feat)
        return obj_feat, cls_feat

## Top-level encoder

In [ ]:
class SimpleObjectCloudEncoder(nn.Module):
    """Stripped-down PTv1-style encoder for semantic object clouds.

    Encodes a variable-size set of categorically-labeled, positioned objects
    into a fixed-size scene summary embedding (CLS).

    Parameters
    ----------
    num_classes   : number of object categories               (default 21)
    d_model       : internal feature dimension                (default 64)
    out_dim       : output CLS embedding dimension            (default 32)
    num_layers    : number of encoder layers L                (default 2)
    rpe_mode      : 'raw' | 'normalized' | 'direction_only' | 'log_distance'
    ffn_expansion : FFN hidden dim multiplier                 (default 2)
    dropout       : dropout rate                              (default 0.0)
    return_obj_features : if True, also return per-object features (B, N, d_model)

    Inputs
    ------
    class_idx : (B, N) int64 tensor of class indices in [0, num_classes).
    obj_pos   : (B, N, 3) float, object centroid positions.
    agent_pos : (B, 3)    float, agent position (origin if using agent-frame coords).
    obj_mask  : (B, N)    bool, True = valid object, False = padding (optional).
                Each sample must have at least one valid object.

    Coordinate convention
    ---------------------
    For object goal navigation, prefer **agent-frame** coordinates::

        obj_pos   = R_agent.T @ (obj_pos_world - agent_pos_world)
        agent_pos = zeros

    The model is agnostic to the chosen frame; this is purely a recommendation.

    Outputs
    -------
    cls_out : (B, out_dim)
    obj_out : (B, N, d_model)   [only if return_obj_features=True]
    """

    def __init__(
        self,
        num_classes: int = 21,
        d_model: int = 64,
        out_dim: int = 32,
        num_layers: int = 2,
        rpe_mode: RPEMode = "log_distance",
        ffn_expansion: int = 2,
        dropout: float = 0.0,
        return_obj_features: bool = False,
    ):
        super().__init__()
        self.d_model = d_model
        self.out_dim = out_dim
        self.rpe_mode = rpe_mode
        self.return_obj_features = return_obj_features

        self.input_embed = ClassEmbedding(num_classes, d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        self.layers = nn.ModuleList([
            SimpleObjectCloudLayer(
                d_model=d_model,
                rpe_mode=rpe_mode,
                ffn_expansion=ffn_expansion,
                dropout=dropout,
            )
            for _ in range(num_layers)
        ])

        self.cls_norm = nn.LayerNorm(d_model)
        self.cls_head = nn.Linear(d_model, out_dim)

    def forward(
        self,
        class_idx: torch.Tensor,                      # (B, N)
        obj_pos: torch.Tensor,                        # (B, N, 3)
        agent_pos: torch.Tensor,                      # (B, 3)
        obj_mask: Optional[torch.Tensor] = None,      # (B, N) bool
    ):
        B, N = class_idx.shape

        obj_feat = self.input_embed(class_idx)                  # (B, N, d)
        cls_feat = self.cls_token.expand(B, 1, self.d_model)    # (B, 1, d)
        cls_pos = agent_pos.unsqueeze(1)                        # (B, 1, 3)

        scene_scale = None
        if self.rpe_mode == "normalized":
            if obj_mask is not None:
                inv_mask = ~obj_mask.unsqueeze(-1)
                pos_for_max = obj_pos.masked_fill(inv_mask, float("-inf"))
                pos_for_min = obj_pos.masked_fill(inv_mask, float("inf"))
                bbox_diag = pos_for_max.max(dim=1).values - pos_for_min.min(dim=1).values
            else:
                bbox_diag = obj_pos.max(dim=1).values - obj_pos.min(dim=1).values
            scene_scale = bbox_diag.norm(dim=-1).clamp(min=1e-6)

        if obj_mask is None:
            obj_mask = torch.ones(B, N, dtype=torch.bool, device=class_idx.device)

        for layer in self.layers:
            obj_feat, cls_feat = layer(
                obj_feat=obj_feat,
                obj_pos=obj_pos,
                cls_feat=cls_feat,
                cls_pos=cls_pos,
                obj_mask=obj_mask,
                scene_scale=scene_scale,
            )

        cls_out = self.cls_head(self.cls_norm(cls_feat.squeeze(1)))  # (B, out_dim)

        if self.return_obj_features:
            return cls_out, obj_feat
        return cls_out

## Sanity check

Builds the encoder with default hyperparameters, runs it on a synthetic batch with variable-size object clouds, and asserts the output shapes.

In [ ]:
torch.manual_seed(0)

B, N = 2, 30
NUM_CLASSES = 21
D_MODEL = 64
OUT_DIM = 32

model = SimpleObjectCloudEncoder(
    num_classes=NUM_CLASSES,
    d_model=D_MODEL,
    out_dim=OUT_DIM,
    num_layers=2,
    rpe_mode="log_distance",
    ffn_expansion=2,
    return_obj_features=True,
)

class_idx = torch.randint(0, NUM_CLASSES, (B, N))
# Agent-frame coords: objects scattered around the agent (origin)
obj_pos = torch.randn(B, N, 3) * 5.0
agent_pos = torch.zeros(B, 3)

# Variable N: second sample only has 12 valid objects
obj_mask = torch.ones(B, N, dtype=torch.bool)
obj_mask[1, 12:] = False

cls_out, obj_out = model(class_idx, obj_pos, agent_pos, obj_mask)

assert cls_out.shape == (B, OUT_DIM), cls_out.shape
assert obj_out.shape == (B, N, D_MODEL), obj_out.shape

n_params = sum(p.numel() for p in model.parameters())
print(f"CLS output shape:        {tuple(cls_out.shape)}")
print(f"Object features shape:   {tuple(obj_out.shape)}")
print(f"CLS output sample norms: {cls_out.norm(dim=-1).tolist()}")
print(f"Total parameters:        {n_params:,}")
print("Sanity check passed.")